# NB_02 · Prétraitement des données
### Prédiction d'annulation de réservations d'hôtel

**Objectif :** appliquer les solutions aux problèmes identifiés dans `NB_01_EDA` (valeurs manquantes,
data leakage, outliers, encodage, normalisation), puis sauvegarder les jeux train/test prêts pour la
modélisation (`NB_03_Dev_Modele`).


In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

df = pd.read_csv('../Donnees/hotel_bookings.csv')
print("Dimensions initiales :", df.shape)


Dimensions initiales : (119390, 32)


## 1. Suppression du data leakage

In [2]:
df_clean = df.drop(columns=['reservation_status', 'reservation_status_date'])
print("Colonnes restantes :", df_clean.shape[1])


Colonnes restantes : 30


**Solution :** `reservation_status` et `reservation_status_date` révèlent directement l'issue de la
réservation. Elles ne seraient jamais connues au moment de la réservation → suppression obligatoire pour
éviter un modèle qui "triche".

## 2. Traitement des valeurs manquantes

In [3]:
# company : ~94% manquant -> colonne supprimée
df_clean = df_clean.drop(columns=['company'])

# agent : NULL = pas d'agent impliqué -> remplacé par 0
df_clean['agent'] = df_clean['agent'].fillna(0)

# country : peu de valeurs manquantes -> remplacées par le mode
df_clean['country'] = df_clean['country'].fillna(df_clean['country'].mode()[0])

# children : 4 lignes seulement -> suppression des lignes
df_clean = df_clean.dropna(subset=['children'])

print("Valeurs manquantes restantes :", df_clean.isnull().sum().sum())
print("Dimensions :", df_clean.shape)


Valeurs manquantes restantes : 0
Dimensions : (119386, 29)


**Solutions appliquées :**
- `company` → colonne supprimée (trop peu d'information exploitable)
- `agent` → 0 (NULL = "aucun agent", catégorie valide)
- `country` → imputation par le mode (impact négligeable)
- `children` → suppression des 4 lignes concernées

## 3. Traitement des doublons

In [4]:
n_dup_before = df_clean.duplicated().sum()
df_clean = df_clean.drop_duplicates()
print(f"Doublons supprimés : {n_dup_before}")
print("Dimensions après suppression des doublons :", df_clean.shape)


Doublons supprimés : 32279
Dimensions après suppression des doublons : (87107, 29)


**Solution :** suppression des lignes strictement identiques. Ces doublons ne représentent pas
des clients distincts identifiables et risquent de sur-pondérer certains profils dans l'apprentissage.

## 4. Traitement des valeurs aberrantes (outliers)

In [5]:
before = df_clean.shape[0]

# adr : suppression de la valeur négative et des valeurs extrêmes (> 1000)
df_clean = df_clean[(df_clean['adr'] >= 0) & (df_clean['adr'] <= 1000)]

# réservations sans aucun occupant : incohérentes -> supprimées
mask_no_guest = (df_clean['adults']==0) & (df_clean['children']==0) & (df_clean['babies']==0)
df_clean = df_clean[~mask_no_guest]

after = df_clean.shape[0]
print(f"Lignes supprimées pour outliers : {before - after}")
print("Dimensions finales avant encodage :", df_clean.shape)


Lignes supprimées pour outliers : 168
Dimensions finales avant encodage : (86939, 29)


**Solutions appliquées :**
- `adr` négatif ou > 1000 : retiré (incohérent avec un tarif journalier réaliste)
- Réservations sans occupant : retirées (incohérentes par nature)
- `lead_time` : valeurs extrêmes conservées (plausibles, informatives)

## 5. Encodage des variables catégorielles

In [6]:
cat_cols = df_clean.select_dtypes(include='object').columns.tolist()
print("Variables catégorielles :", cat_cols)
print("Colonnes avant encodage :", df_clean.shape[1])

df_encoded = pd.get_dummies(df_clean, columns=cat_cols, drop_first=True)
print("Colonnes après One-Hot Encoding :", df_encoded.shape[1])


Variables catégorielles : ['hotel', 'arrival_date_month', 'meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type', 'assigned_room_type', 'deposit_type', 'customer_type']
Colonnes avant encodage : 29
Colonnes après One-Hot Encoding : 244


/tmp/ipykernel_591/1839725795.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df_clean.select_dtypes(include='object').columns.tolist()


**Solution :** **One-Hot Encoding** pour toutes les variables catégorielles nominales
(pas d'ordre naturel entre les catégories → éviter le Label Encoding qui créerait un ordre artificiel).

## 6. Séparation features / cible et split train / test

In [7]:
X = df_encoded.drop(columns=['is_canceled'])
y = df_encoded['is_canceled']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)
print("Proportion classe 1 (train) :", y_train.mean().round(3))
print("Proportion classe 1 (test)  :", y_test.mean().round(3))


X_train : (69551, 243)
X_test  : (17388, 243)
Proportion classe 1 (train) : 0.273
Proportion classe 1 (test)  : 0.273


**Solution :** split **80 % / 20 %**, `random_state=42` (reproductibilité), `stratify=y`
(conserver la même proportion d'annulations dans train et test).

## 7. Normalisation

In [8]:
num_cols = ['lead_time','stays_in_weekend_nights','stays_in_week_nights','adults',
            'children','babies','adr','booking_changes','days_in_waiting_list',
            'previous_cancellations','total_of_special_requests']

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

X_train_scaled[num_cols].describe().T[['mean','std']]


,mean,std
lead_time,-4.157969e-17,1.000007
stays_in_weekend_nights,-6.946975e-18,1.000007
stays_in_week_nights,1.070651e-16,1.000007
adults,1.269866e-16,1.000007
children,-2.502954e-17,1.000007
babies,1.838905e-17,1.000007
adr,-1.254542e-16,1.000007
booking_changes,9.194526e-18,1.000007
days_in_waiting_list,-8.581557e-18,1.000007
previous_cancellations,1.348530e-17,1.000007


**Solution :** `StandardScaler` entraîné uniquement sur `X_train` puis appliqué à `X_test`
(pas de fuite d'information). Utile notamment pour la régression logistique.

## 8. Sauvegarde des jeux de données et du scaler

In [9]:
os.makedirs('../Donnees/processed', exist_ok=True)
os.makedirs('../Modeles', exist_ok=True)

X_train_scaled.to_csv('../Donnees/processed/X_train.csv', index=False)
X_test_scaled.to_csv('../Donnees/processed/X_test.csv', index=False)
y_train.to_csv('../Donnees/processed/y_train.csv', index=False)
y_test.to_csv('../Donnees/processed/y_test.csv', index=False)

joblib.dump(scaler, '../Modeles/scaler.pkl')
joblib.dump(list(X_train_scaled.columns), '../Modeles/feature_columns.pkl')

print("Fichiers sauvegardés dans ../Donnees/processed/ et ../Modeles/")


Fichiers sauvegardés dans ../Donnees/processed/ et ../Modeles/


## Résumé

| Étape | Avant | Après |
|---|---|---|
| Lignes | 119 390 | voir dimensions ci-dessus |
| Colonnes | 32 | 30 (leakage + company retirés), puis ~245 après encodage |
| Valeurs manquantes | 4 colonnes concernées | 0 |
| Doublons | ~27 % | 0 |

➡️ Suite dans **`NB_03_Dev_Modele.ipynb`** : entraînement des modèles de classification.
